In [ ]:
from pystac.client import Client
from odc.stac import load

from odc.geo.geom import BoundingBox

In [ ]:
# PNG
# ll = -9.0, 143.0
# ur = -7.2, 145.5

# Vanuatu
# ll = -16.57, 167.6
# ur = -16.36, 167.87

# Nukuira
ll = -16.576, 178.691
ur = -16.462, 178.852
bbox = BoundingBox(left=ll[1], bottom=ll[0], right=ur[1], top=ur[0], crs="EPSG:4326")

bbox.explore()

In [ ]:
client = Client.open("https://stac.digitalearthpacific.org")

items = client.search(collections=["dep_s2_mangroves"], intersects=bbox.polygon).item_collection()

print(f"Found {len(items)} items.")

In [ ]:
data = load(items, intersects=bbox.polygon)

# Calculate area per year, at 10 m resolution, where mangrove values are either 1 or 2
mangrove_area = ((data == 1) | (data == 2)).sum(dim=["x", "y"]) * 10 * 10
mangrove_area

In [ ]:
masked = data.where((data != 255) & (data != 0))

masked.mangroves.plot.imshow(col="time", col_wrap=4, cmap="viridis_r", vmin=0, vmax=2, xticks=[], yticks=[], add_colorbar=False)

In [ ]:
# Plot mangrove area over time with a tight y-range and relative change
import matplotlib.pyplot as plt

area_km2 = mangrove_area.mangroves / 1_000_000
years = mangrove_area.time.values
baseline = area_km2.isel(time=0)
pct_change = (area_km2 / baseline - 1) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years, area_km2.values, marker="o", linewidth=2, color="#1f77b4", label="Area (km²)")

# Tighten y-limits so subtle changes are visible
ymin = float(area_km2.min())
ymax = float(area_km2.max())
pad = (ymax - ymin) * 0.15 if ymax > ymin else 0.05
ax.set_ylim(ymin - pad, ymax + pad)

ax.set_xlabel("Year")
ax.set_ylabel("Mangrove area (km²)", color="#1f77b4")
ax.tick_params(axis="y", labelcolor="#1f77b4")
ax.grid(True, alpha=0.3)

ax2 = ax.twinx()
ax2.plot(years, pct_change.values, marker="s", linestyle="--", linewidth=1.5, color="#ff7f0e", label="Change vs first year (%)")
ax2.set_ylabel("Change vs first year (%)", color="#ff7f0e")
ax2.tick_params(axis="y", labelcolor="#ff7f0e")

ax.set_title("Mangrove Area Over Time")
fig.tight_layout()
plt.show()